# Latent forecasting on Lorenz-63, multiple series: is the tradeoff dissolvable?

The autoencoder experiment (`NeuralNetworkApproximation.ipynb`) showed the bounded/unbounded
chain reconstructing at the noise floor. `LatentForecastComparison.ipynb` added the forecasting
arm on a single 20,000-step trajectory. This notebook asks the same question with the training
pool widened to **multiple independent trajectories**, to see whether the result was an artefact
of having only one series to fit.

**The claim under test.** Reconstruction fidelity and long-horizon forecast quality are widely
treated as competing objectives to be *balanced* — by weighted losses (Coupled Attention
Networks, MemAAE, CAAE), by spectral truncation (SSP, 2026), by shared encoders. The claim here
is that this tension is a suboptimal operating point rather than a law, and that separating the
two latents *dissolves* it.

**Two data pools, same observation space.** A training pool of several independent Lorenz
trajectories, each split chronologically 70/15/15 exactly as before and then pooled across
series; and a second, disjoint pool of trajectories the model never sees in any form during
training, used purely for evaluation — no temporal split, since none of it was ever training
data to begin with. See the data section below for the exact construction.

**Three models, identical footing.**

| | Model | What it says about the tradeoff |
|---|---|---|
| **A** | Ours: bounded/unbounded chain, one forecaster | dissolve it — separate carriers, separate stages |
| **B** | Weighted-loss AE+GRU, phi swept 0 to 1 | balance it — one latent, one knob |
| **C** | Physics-informed latent AE (PINN) | sidestep it — let the equations supply the dynamics |

Model B is the important one. Sweeping phi traces the empirical reconstruction-forecast Pareto
front, which is what turns every other model in the comparison from an isolated number into a
point relative to a curve.


# Imports

In [ ]:
import sys
import time
from pathlib import Path

# Notebook lives in Experimentation/, so add the repo root to sys.path
ROOT = Path.cwd().parent if Path.cwd().name == "Experimentation" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import clear_output

from Src import DecoupledModel
from Comp import JointAEGRU, JointAEGRUSigmoid, PhysicsLatentAE
from Utils import (LoadLorenzMultiSeries, PALETTE, DPI, PlotNFeatSeries, PlotLossCurves,
                   PlotParity, PlotChannelBar, PlotAttractor3D, PlotMatrixGrid,
                   PlotHorizonCurves, PlotParetoFront, PlotTimeToQuality)
from Utils.Checkpoints import TrainOrLoad
from Utils.Metrics import (R2, PerChannelR2, NRMSE, PerHorizonNRMSE, VPT,
                           BoundViolation, Saturation, AttractorStats, ZMaxMap,
                           FitStateMap, ApplyStateMap, StepsPerLyapunov, LAMBDA_MAX,
                           Mse as MseNp)
from Utils.Rollout import (EvaluateModel, ReconMetrics, ForecastMetrics,
                           PersistenceRollout, MeanRollout, EvalWindowsFromTrajectories,
                           MakeSeriesWindows)
from Utils.Benchmark import (CountParams, CountInferenceParams, CountFlops, MatchWidth,
                             EpochTrain, TimeToTarget, TimeRollout, PinThreads, SeedAll)


# VARS

In [ ]:
DATAPATH = ROOT / "Data" / "LorenzLiftMulti.npz"

# Regenerate with:
#     python -c "from SyntheticGenerators.LorenzLift import save_multi_series_dataset as s; s()"
assert DATAPATH.exists(), (f"missing {DATAPATH} -- see "
                           "SyntheticGenerators/LorenzLift.py:save_multi_series_dataset")
print(DATAPATH.name, "found")


# Data Exp

In [ ]:
Multi = LoadLorenzMultiSeries(DATAPATH)
TrainSeries, ValSeries, TestSeries = Multi["train"], Multi["val"], Multi["test"]
CleanSeries, StatesSeries = Multi["clean"], Multi["states"]
HoldoutObs, HoldoutStates = Multi["holdout_obs"], Multi["holdout_states"]
N_SERIES = TrainSeries.shape[0]

# Pooled across every training-pool series -- same role as the single-series
# train/val/test/states/clean, just concatenated over the series axis. Stage A,
# the probes and the aggregate metrics train and score on these.
train = TrainSeries.reshape(-1, TrainSeries.shape[-1])
val = ValSeries.reshape(-1, ValSeries.shape[-1])
test = TestSeries.reshape(-1, TestSeries.shape[-1])
clean = CleanSeries.reshape(-1, CleanSeries.shape[-1])
states = StatesSeries.reshape(-1, StatesSeries.shape[-1])

_I, _J = TrainSeries.shape[1], TrainSeries.shape[1] + ValSeries.shape[1]
StatesTrain = StatesSeries[:, :_I].reshape(-1, 3)
StatesVal = StatesSeries[:, _I:_J].reshape(-1, 3)
StatesTest = StatesSeries[:, _J:].reshape(-1, 3)

# Series 0 only, kept temporally contiguous. Every plot below that draws a
# single time series or attractor uses this one trajectory rather than the
# pooled arrays, which jump between trajectories at each series boundary.
train0, val0, test0 = TrainSeries[0], ValSeries[0], TestSeries[0]
StatesTrain0, StatesVal0, StatesTest0 = (StatesSeries[0, :_I], StatesSeries[0, _I:_J],
                                         StatesSeries[0, _J:])

print(f"train {TrainSeries.shape}  val {ValSeries.shape}  test {TestSeries.shape}  "
      f"({N_SERIES} training-pool series, pooled to {train.shape[0]:,} timesteps)")
print(f"clean {CleanSeries.shape}  states {StatesSeries.shape}")
print(f"holdout trajectories {HoldoutObs.shape}  states {HoldoutStates.shape}  "
      f"(never trained on, no temporal split)")
# --- Task 2 verification: no split is standardised with its own statistics ---
# Every mean/std constant used to build this .npz (state, clean-signal, and
# final observation scale) is fit on the pooled TRAIN portion of the training
# trajectories only (SyntheticGenerators/LorenzLift.py:make_multi_series_dataset)
# and applied unchanged to val, test, and holdout. So train lands at exactly
# 0/1 by construction; val, test and holdout should be CLOSE to 0/1 but not
# exactly it -- if holdout in particular came out exactly 0/1, it would mean
# it was standardised with its own statistics rather than train's, i.e. the
# leak is still present.
print("per-split mean/std (train is exact 0/1 by construction; the rest must not be):")
for _name, _arr in [("train", train), ("val", val), ("test", test), ("holdout", HoldoutObs)]:
    print(f"  {_name:8s} mean {_arr.mean():+.6f}  std {_arr.std():.6f}")
assert abs(train.mean()) < 1e-8 and abs(train.std() - 1) < 1e-8, "train is not exactly standardised"
assert abs(HoldoutObs.mean()) > 1e-4 or abs(HoldoutObs.std() - 1) > 1e-4, (
    "holdout is exactly 0/1 -- it was standardised with its own statistics, leak still present")
print("PASS: train is exact 0/1; val/test/holdout are close but not exact -- no split's own")
print("      statistics leaked into the constants used to build it.")


In [ ]:
# The observations the models actually see, series 0 of the training pool.
Fig, Axes = PlotNFeatSeries(
    train0[:1200][None], Cols=5,
    Title=f"Train observations, series 0 of {N_SERIES}, first 1200 steps",
    YLabel="Standardised value")
plt.show()


In [ ]:
# The 3-d truth underneath, series 0. Never in a loss -- diagnostics only.
Fig, Axes = PlotNFeatSeries(
    StatesTrain0[:1200][None], Cols=3, Titles=["$x$", "$y$", "$z$"],
    Title=f"True Lorenz-63 state, series 0 of {N_SERIES}, first 1200 steps "
          "(diagnostics only, never in a loss)",
    YLabel="State value")
plt.show()

Fig, Axes = PlotAttractor3D(
    [StatesTrain0[:6000]], ["True attractor (train, series 0)"], Width=4.2, Height=4.2,
    Title="Lorenz-63 attractor, training split, series 0")
plt.show()


In [ ]:
Ranges = pd.DataFrame({"min": train.min(0), "max": train.max(0),
                       "std": train.std(0)}).round(3)
print(f"observation channels: {len(Ranges)},  global range "
      f"[{train.min():.2f}, {train.max():.2f}]  (pooled across {N_SERIES} series)")
Fig, Ax = PlotChannelBar(
    train.std(0), Title=f"Per-channel standard deviation (train, pooled across {N_SERIES} series)",
    YLabel="Standard deviation", Baseline=1.0, BaselineLabel="unit std (standardisation target)")
plt.show()
Ranges.T


In [ ]:
# Sample of the held-out trajectories -- never trained on in any form, these are
# what the long rollouts are scored against.
Fig, Axes = PlotNFeatSeries(
    HoldoutObs[:3, :800, :6], Cols=3,
    Labels=[f"holdout trajectory {i}" for i in range(3)],
    Title="Three held-out trajectories, never trained on, first 6 channels",
    YLabel="Standardised value")
plt.show()


Why two pools instead of one train/test split. A single trajectory's test split, even a
long one, is a continuation of the same orbit the model trained on -- it shares the attractor's
local geometry near wherever training happened to stop. The training pool here is **several
independent trajectories**, each individually split 70/15/15 like the single-series notebook,
then pooled; that already tests whether the architecture generalises across starting conditions
rather than memorising one orbit.

`HoldoutObs` goes a step further: these trajectories are **never split, never touched during
training at all** -- not even their first steps. They exist purely to answer "does this transfer
to a trajectory the model has literally never seen", which a temporal split alone cannot answer.
Independent trajectories also fix a second problem: long rollouts cut from one continuous test
series overlap heavily and share most of their future, badly understating the spread across
"different" starts.

Known caveat, recorded and not fixed: `lift()` standardises the shared nonlinearity's inner
statistics using full-series data before any split, a mild leak baked into the generator upstream
of the otherwise train-only observation scaling. Left alone so these numbers stay structurally
comparable to the single-series notebook.


# Config

In [ ]:
N = train.shape[-1]     # observation dim, 30
K = 3                   # unbounded forecast latent, = known intrinsic dim of Lorenz-63
A, B = 8, 8             # bounded latent C_t is A x B in [0, 1]

DT = 0.01
STEPS_PER_LYAP = StepsPerLyapunov(DT)      # ~110
WARM = 64               # lookback, matches the generator's windows(lookback=64)
UNROLL = 20             # training rollout depth
HORIZON = 550           # ~5 Lyapunov times -- the main forecast curve
CLIMATE = 5500          # ~50 Lyapunov times -- attractor statistics
THRESHOLD = 0.4         # NRMSE level at which VPT is read off

# -- Protocol: matched EPOCHS, not matched wall clock. Every parameter group
# gets exactly EPOCHS passes over its own training data -- Stage A over the
# per-timestep pool, Stage B over the b-space windows, the joint baselines
# over their sequence windows. This is reproducible across machines, unlike a
# wall-clock budget, and it removes batch size and thread count as fairness
# variables: a bigger batch makes an epoch faster, it does not change how many
# passes over the data a model gets. MARKS are epoch numbers at which every
# model's metrics are snapshotted, giving the whole compute-scaling curve from
# one run instead of training separately at a second budget.
QUICK = False            # True -> a handful of epochs, for a smoke test
EPOCHS = 5 if QUICK else 60
MARKS = (1, 2, 5) if QUICK else (5, 10, 20, 40, 60)
SEEDS = [0] if QUICK else [0, 1, 2]
PHIS = [0.0, 0.25, 0.75, 1.0] if QUICK else [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]

# Batch sizes, tuned per model group by measured throughput. Fairness-neutral
# now that the protocol counts epochs rather than seconds.
BATCH_STAGE_A = 4096     # Ours Stage A, per-timestep autoencoder (was BATCH=256)
BATCH_STAGE_B = 1024     # Ours Stage B, b-space sequences (was SEQ_BATCH=64)
BATCH_SEQ = 256          # joint / sigmoid / ours-joint sequence training (was SEQ_BATCH=64)
BATCH_PINN = 4096        # PINN, per-timestep triples (was BATCH=256)
SEQ_STRIDE = 4
LR = 1e-3
W_CONSIST = 1.0
LAM_PHYS = 1.0
PLOT_EVERY = 10
NOISE_FLOOR = float(Multi["noise_floor_mse"])   # measured, not the naive noise**2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THREADS = PinThreads(8)   # speed only -- epochs, not seconds, define the comparison

# Every trained model is cached here by name -- see Utils/Checkpoints.TrainOrLoad.
# Delete a file (or the whole directory) to force that model to retrain; nothing
# below re-trains a model whose checkpoint is already on disk. The epoch-matched
# protocol invalidates every checkpoint trained under the old wall-clock
# protocol -- Checkpoints/multi_series/ must be deleted entirely before the
# first run under this config.
CHECKPOINT_DIR = ROOT / "Checkpoints" / "multi_series"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"device {DEVICE}, torch threads {THREADS}")
print(f"{EPOCHS} epochs/model, marks {MARKS}, seeds {SEEDS}, phi {PHIS}")
print(f"{STEPS_PER_LYAP:.1f} steps per Lyapunov time -> "
      f"horizon {HORIZON} = {HORIZON / STEPS_PER_LYAP:.1f} LT, "
      f"climate {CLIMATE} = {CLIMATE / STEPS_PER_LYAP:.0f} LT")


In [ ]:
# --- Task 1 verification: NOISE_FLOOR is measured, not the naive noise**2 ---
# noise_floor_mse (SyntheticGenerators/LorenzLift.py:make_multi_series_dataset) is
# computed directly from the realised noise draw in the final standardised scale
# (((raw_obs - clean) / obs_sd) ** 2).mean() on the pooled TRAIN split -- not
# assumed from the noise parameter. This was already the case before this session;
# recorded here explicitly so it is auditable rather than asserted.
NaiveFloor = 0.05 ** 2
ClosedFormFloor = 0.05 ** 2 / (1 + 0.05 ** 2)   # the (obs_sd ~ 1) approximation
print(f"NOISE_FLOOR (measured, used everywhere below) : {NOISE_FLOOR:.6f}")
print(f"naive noise**2                                : {NaiveFloor:.6f}")
print(f"closed-form noise**2/(1+noise**2)              : {ClosedFormFloor:.6f}")
assert abs(NOISE_FLOOR - NaiveFloor) > 1e-6,     "NOISE_FLOOR equals the naive constant -- the empirical measurement is not wired in"
print("NOISE_FLOOR is a measured quantity, distinct from both closed-form estimates.")


The orphaned `windows(horizon=50)` default in the generator is only ~0.45 Lyapunov times — far
too short to expose the failure modes at issue here. Everything below runs to 5 Lyapunov times
for the error curve and 50 for the climate statistics.

## Data

In [ ]:
XTrain = torch.tensor(train, dtype=torch.float32, device=DEVICE)
XVal = torch.tensor(val, dtype=torch.float32, device=DEVICE)
XTest = torch.tensor(test, dtype=torch.float32, device=DEVICE)

Loader = DataLoader(TensorDataset(XTrain), batch_size=BATCH_STAGE_A, shuffle=True, drop_last=True)

# One climatological scale for everyone, so the 0.4 VPT threshold means the
# same thing across models. Both come from the held-out pool -- never the
# training pool -- exactly as in the single-series notebook.
SCALE = float(HoldoutObs.std())
BOUNDS = (float(train.min()), float(train.max()))

# Held-out rollout windows, from trajectories never seen in training at all.
WarmNp, FutNp = EvalWindowsFromTrajectories(HoldoutObs, WARM, HORIZON, per_traj=2, seed=7)
Warm = torch.tensor(WarmNp, dtype=torch.float32, device=DEVICE)
Fut = torch.tensor(FutNp, dtype=torch.float32, device=DEVICE)

WarmCNp, FutCNp = EvalWindowsFromTrajectories(HoldoutObs, WARM, CLIMATE, per_traj=1, seed=11)
WarmC = torch.tensor(WarmCNp, dtype=torch.float32, device=DEVICE)
FutC = torch.tensor(FutCNp, dtype=torch.float32, device=DEVICE)

print(f"forecast windows {tuple(Warm.shape)} -> {tuple(Fut.shape)}")
print(f"climate windows  {tuple(WarmC.shape)} -> {tuple(FutC.shape)}")
print(f"scale {SCALE:.4f}   bounds {BOUNDS[0]:.2f} .. {BOUNDS[1]:.2f}")


In [ ]:
# Training windows, one series at a time so none crosses a trajectory boundary --
# pooling first and sliding a window over the concatenation would splice the
# tail of one series onto the head of the next. Stage B never touches
# observations at all -- it works on the b series alone -- so only the shapes
# here matter to it, not XFull itself.
XFullNp = MakeSeriesWindows(TrainSeries, WARM + UNROLL, stride=SEQ_STRIDE)
XFull = torch.tensor(XFullNp, dtype=torch.float32, device=DEVICE)

# Same windows on val, capped per series so the total stays close to the
# single-series notebook's count -- this is a cheap in-training score, not the
# real evaluation.
XValFullNp = MakeSeriesWindows(ValSeries, WARM + UNROLL, stride=64, max_per_series=8)
XValFull = torch.tensor(XValFullNp, dtype=torch.float32, device=DEVICE)
XValWarm, XValFut = XValFull[:, :WARM], XValFull[:, WARM:]

print(f"{XFull.shape[0]} training windows of {WARM}+{UNROLL} from {N_SERIES} series, "
      f"{XValFull.shape[0]} val windows")
print(f"XFull {tuple(XFull.shape)}  ({XFull.numel() * 4 / 1e6:.0f} MB)")


## Matched parameter budget

Ours sets the budget; the baselines are fitted to it by bisecting their hidden width. Exact
matching is impossible — parameter counts jump in steps of a few hundred — so the target is
within a couple of percent and the actual counts are printed rather than rounded away.

Two counts are reported. `LatentBounded` is a **training-only teacher**: it supplies the target
for `m(b_t)` and never runs at inference. Counting it inside a training budget is right;
counting it in a speed table would understate the model.

In [ ]:
SeedAll(0)
Reference = DecoupledModel(N, K, A, B)
TARGET = CountParams(Reference)

WidthJoint = MatchWidth(lambda w: JointAEGRU(N, latent=16, width=w), TARGET)
WidthSigmoid = MatchWidth(lambda w: JointAEGRUSigmoid(N, latent=16, width=w), TARGET)
WidthPinn = MatchWidth(lambda w: PhysicsLatentAE(N, width=w), TARGET)

Budget = pd.DataFrame([
    {"model": "Ours (A)", "width": "-", "trainable": TARGET,
     "inference": CountInferenceParams(Reference), "flops": CountFlops(Reference)},
    {"model": "Joint AE+GRU (B)", "width": WidthJoint,
     "trainable": CountParams(JointAEGRU(N, latent=16, width=WidthJoint)),
     "inference": CountInferenceParams(JointAEGRU(N, latent=16, width=WidthJoint)),
     "flops": CountFlops(JointAEGRU(N, latent=16, width=WidthJoint))},
    {"model": "Joint AE+GRU, sigmoid latent (D)", "width": WidthSigmoid,
     "trainable": CountParams(JointAEGRUSigmoid(N, latent=16, width=WidthSigmoid)),
     "inference": CountInferenceParams(JointAEGRUSigmoid(N, latent=16, width=WidthSigmoid)),
     "flops": CountFlops(JointAEGRUSigmoid(N, latent=16, width=WidthSigmoid))},
    {"model": "Physics latent AE (C)", "width": WidthPinn,
     "trainable": CountParams(PhysicsLatentAE(N, width=WidthPinn)),
     "inference": CountInferenceParams(PhysicsLatentAE(N, width=WidthPinn)),
     "flops": CountFlops(PhysicsLatentAE(N, width=WidthPinn))},
])
Budget["err %"] = (100 * (Budget["trainable"] - TARGET) / TARGET).round(2)

print(f"Stage A {sum(CountParams(m) for m in Reference.StageAModules()):,}  "
      f"Stage B {sum(CountParams(m) for m in Reference.StageBModules()):,}")
Budget

## Loss

Each epoch function runs exactly one full pass over its own training data and returns. There is
no wall-clock deadline to check inside the batch loop -- the protocol is matched epochs, so every
parameter group gets the same number of passes regardless of how expensive its batches are.

In [13]:
MseT = nn.MSELoss()


def StageAEpoch(Model, Opt):
    """Per-timestep autoencoder, identical to the objective in the existing notebook.

    CTeach is deliberately not detached and R stays in the optimiser, matching
    NeuralNetworkApproximation.ipynb so the two runs are comparable. It does mean
    the consistency term pulls teacher toward student as much as the reverse.
    """
    Sums = [0.0, 0.0, 0.0]
    Nb = 0
    Batches = tqdm(Loader, desc="stage A", leave=False)
    for (x,) in Batches:
        XHat, Ct, CTeach, _, _ = Model(x)
        Recon, Consist = MseT(XHat, x), MseT(Ct, CTeach)
        Total = Recon + W_CONSIST * Consist
        Opt.zero_grad(); Total.backward(); Opt.step()
        Vals = [Total.item(), Recon.item(), Consist.item()]
        Sums = [s + v for s, v in zip(Sums, Vals)]
        Nb += 1
        Batches.set_postfix(recon=f"{Vals[1]:.2e}", consist=f"{Vals[2]:.2e}")
    Nb = max(Nb, 1)
    return {"total": Sums[0] / Nb, "recon": Sums[1] / Nb, "consistency": Sums[2] / Nb}


def StageBEpoch(Model, Opt, Bwarm, Bfut):
    """The single forecaster, on latents from the frozen Stage-A encoder.

    The loss lives entirely in b-space: the encoder's own b_t are the targets, so
    nothing downstream of f can shape the forecaster. At rollout time C_{t+1} =
    m(b_{t+1}) reuses the same static m that reconstruction uses, so g is the only
    learned dynamics in the chain.

    Note the units. This number is an MSE on b, not on x, so it is not comparable
    to the phi baselines' fcst or to Stage A's recon. val_fcst, which comes from
    ValScores -> Model.Rollout, is still in observation space and is the number to
    read across models.
    """
    Perm = torch.randperm(Bwarm.shape[0])
    Sum, Nb = 0.0, 0
    Batches = tqdm(range(0, len(Perm), BATCH_STAGE_B), desc="stage B", leave=False)
    for i in Batches:
        j = Perm[i:i + BATCH_STAGE_B]
        b0, Bt = Bwarm[j], Bfut[j]
        _, St = Model.G(b0[:, :-1], None)
        Bc, Out = b0[:, -1], []
        for _ in range(UNROLL):
            Bc, St = Model.G.Step(Bc, St)
            Out.append(Bc)
        Loss = MseT(torch.stack(Out, 1), Bt)
        Opt.zero_grad(); Loss.backward(); Opt.step()
        Sum += Loss.item(); Nb += 1
        Batches.set_postfix(fcst=f"{Loss.item():.2e}")
    return {"fcst": Sum / max(Nb, 1)}


def JointEpoch(Model, Opt, Phi):
    Perm = torch.randperm(XFull.shape[0])
    Sums, Nb = [0.0, 0.0], 0
    Batches = tqdm(range(0, len(Perm), BATCH_SEQ), desc=f"phi {Phi:g}", leave=False)
    for i in Batches:
        Rec, Fc = Model.Losses(XFull[Perm[i:i + BATCH_SEQ]], WARM)
        Loss = (1.0 - Phi) * Rec + Phi * Fc
        Opt.zero_grad(); Loss.backward(); Opt.step()
        Sums = [Sums[0] + Rec.item(), Sums[1] + Fc.item()]
        Nb += 1
        Batches.set_postfix(recon=f"{Rec.item():.2e}", fcst=f"{Fc.item():.2e}")
    Nb = max(Nb, 1)
    return {"recon": Sums[0] / Nb, "fcst": Sums[1] / Nb}


def PinnEpoch(Model, Opt, T3):
    Perm = torch.randperm(T3.shape[0])
    Sums, Nb = [0.0, 0.0], 0
    Batches = tqdm(range(0, len(Perm), BATCH_PINN), desc="pinn", leave=False)
    for i in Batches:
        Rec, Phys = Model.Losses(T3[Perm[i:i + BATCH_PINN]])
        Loss = Rec + LAM_PHYS * Phys
        Opt.zero_grad(); Loss.backward(); Opt.step()
        Sums = [Sums[0] + Rec.item(), Sums[1] + Phys.item()]
        Nb += 1
        Batches.set_postfix(recon=f"{Rec.item():.2e}", phys=f"{Phys.item():.2e}")
    Nb = max(Nb, 1)
    return {"recon": Sums[0] / Nb, "physics": Sums[1] / Nb}


In [14]:
@torch.no_grad()
def ValScores(Model, Fcst=True, **Kw):
    """Cheap in-training validation. EpochTrain tracks this time separately."""
    Was = Model.training
    Model.eval()
    Out = {"val_recon": MseT(Model.Reconstruct(XValFull), XValFull).item()}
    if Fcst:
        Out["val_fcst"] = MseT(Model.Rollout(XValWarm, UNROLL, **Kw), XValFut).item()
    if Was:
        Model.train()
    return Out


def LossPlot(Keys, Floor=None, Title=None):
    """Live loss curve for EpochTrain, in the style of the autoencoder notebook."""
    def Fn(Hist):
        PlotLossCurves({k: Hist[k] for k in Keys if k in Hist},
                       Floor=Floor, Title=Title)
    return Fn

# Model A -- ours, two-stage

Stage A trains the autoencoder exactly as in `NeuralNetworkApproximation.ipynb`. Stage B then
**freezes** `E, f, m, r, D`, runs the frozen encoder over every training timestamp to get the
`b_t` series, and fits one forecaster to it:

```
b_{t+1} = g(b_t)          GRU on the unbounded latent, the only learned dynamics
C_{t+1} = m(b_{t+1})      the SAME static m reconstruction uses -- recomputed, not advanced
x_{t+1} = D(C_{t+1})
```

Stage B's loss is an MSE on `b` against the encoder's own `b_t`, so the forecast gradient never
reaches `m` or `D`. `C` still lands in `[0, 1]` because `m` ends in a sigmoid, so boundedness
survives an arbitrarily long rollout however far `b` drifts.

Freezing is what makes the decoupling real here. The topology is a *chain* -- reconstruction
gradients flow through `b` -- so if both stages trained together the forecast objective would
reshape `b` and therefore reconstruction, which is precisely the coupling the phi baseline
represents. Decoupling in this architecture comes from the training schedule, and the joint
variant in the ablations is run specifically to show what happens without it.

Stage B trains on **precomputed** latents with no decoder in the graph, so its updates are far
cheaper than the joint model's, which re-encodes every batch. That is a genuine computational
advantage of two-stage decoupling rather than an artifact -- at matched epochs it shows up as
cheaper wall clock per epoch, not as extra passes over the data.


## Training

In [ ]:
def TrainOurs(Seed, Epochs=EPOCHS, Verbose=True):
    def Build():
        SeedAll(Seed)
        return DecoupledModel(N, K, A, B).to(DEVICE)

    def Train(Model):
        return _TrainOursBody(Model, Seed, Epochs, Verbose)

    return TrainOrLoad(CHECKPOINT_DIR / f"ours_seed{Seed}.pt", Build, Train, DEVICE)


def _TrainOursBody(Model, Seed, Epochs, Verbose):
    # -- Stage A: reconstruction only. EPOCHS passes over the per-timestep pool.
    OptA = torch.optim.Adam(Model.parameters(), lr=LR)
    HistA = EpochTrain(lambda: StageAEpoch(Model, OptA),
                       EvalFn=lambda: ValScores(Model, Fcst=False),
                       epochs=Epochs, marks=MARKS, name=f"ours A (seed {Seed})",
                       PlotFn=LossPlot(["total", "recon", "consistency", "val_recon"],
                                       Floor=NOISE_FLOOR,
                                       Title=f"Stage A, seed {Seed}") if Verbose else None,
                       plot_every=PLOT_EVERY, verbose=Verbose)

    # -- Freeze, then run the encoder over every timestamp once, one series at a
    # time so Stage B's windows never cross a trajectory boundary either.
    for m in (Model.E, Model.F, Model.M, Model.D, Model.R):
        for p in m.parameters():
            p.requires_grad_(False)
    with torch.no_grad():
        BsSeries = (Model.F(Model.E(XTrain))
                   .view(N_SERIES, TrainSeries.shape[1], K).cpu().numpy())
    BsWindows = MakeSeriesWindows(BsSeries, WARM + UNROLL, stride=SEQ_STRIDE)
    Bwarm = torch.tensor(BsWindows[:, :WARM], dtype=torch.float32, device=DEVICE)
    Bfut = torch.tensor(BsWindows[:, WARM:], dtype=torch.float32, device=DEVICE)

    # -- Stage B: the one forecaster. Same EPOCHS passes, over its own data.
    OptB = torch.optim.Adam(Model.G.parameters(), lr=LR)
    HistB = EpochTrain(lambda: StageBEpoch(Model, OptB, Bwarm, Bfut),
                       EvalFn=lambda: ValScores(Model),
                       epochs=Epochs, marks=MARKS, name=f"ours B (seed {Seed})",
                       PlotFn=LossPlot(["fcst", "val_fcst", "val_recon"],
                                       Title=f"Stage B, seed {Seed}") if Verbose else None,
                       plot_every=max(PLOT_EVERY // 2, 1), verbose=Verbose)

    Hist = {"elapsed": list(HistA["elapsed"])
                       + [HistA["total_time"] + t for t in HistB["elapsed"]],
            "val_recon": HistA["val_recon"] + HistB["val_recon"],
            "total_time": HistA["total_time"] + HistB["total_time"],
            "stageA": HistA, "stageB": HistB}
    return Hist


OursModels, OursHists = [], []
for Seed in SEEDS:
    Model, Hist = TrainOurs(Seed, Verbose=(Seed == SEEDS[0]))
    OursModels.append(Model); OursHists.append(Hist)

print(f"\nstage A {OursHists[0]['stageA']['epochs_done']} epochs, "
      f"stage B {OursHists[0]['stageB']['epochs_done']} epochs, "
      f"{OursHists[0]['total_time']:.1f}s total")


## Final losses

In [ ]:
HA, HB = OursHists[0]["stageA"], OursHists[0]["stageB"]
Fig, Ax = PlotLossCurves(
    {k: HA[k] for k in ("total", "recon", "consistency", "val_recon")},
    Floor=NOISE_FLOOR, Title="Stage A: autoencoder training, seed 0")
plt.show()

# fcst is an MSE on b, val_* are MSEs on x -- different units, hence the note.
Fig, Ax = PlotLossCurves(
    {k: HB[k] for k in ("fcst", "val_fcst", "val_recon")},
    Title="Stage B: forecaster on frozen latents, seed 0\n"
          "(fcst is MSE on $b$; val_fcst / val_recon are MSE on $x$)",
    YLabel="Loss (MSE, mixed units)")
plt.show()

for Name, V in [("total", HA["total"][-1]), ("recon", HA["recon"][-1]),
                ("consistency", HA["consistency"][-1]), ("val_recon", HA["val_recon"][-1]),
                ("noise floor", NOISE_FLOOR), ("stage B fcst", HB["fcst"][-1]),
                ("stage B val_fcst", HB["val_fcst"][-1]),
                ("stage B val_recon", HB["val_recon"][-1])]:
    print(f"{Name:>18}: {V:.5f}")

Sanity gate: Stage A should land close to the 0.0025 noise floor (R^2 ~0.997), as in the
single-series experiment. The obs scale here is fit on the pooled multi-series training pool
rather than one trajectory, so the exact MSE need not match that run's number -- the ratio to
its own floor should. If it does not, something upstream has broken and nothing downstream is
worth reading.


## True vs predicted observations

In [ ]:
Best = OursModels[0]
with torch.no_grad():
    XHatTest = Best.Reconstruct(XTest.unsqueeze(0)).squeeze(0).cpu().numpy()
    XHatTest0 = Best.Reconstruct(
        torch.tensor(test0, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    ).squeeze(0).cpu().numpy()

print(f"test recon MSE {MseNp(test, XHatTest):.5f}   "
      f"(pooled across {N_SERIES} series, floor {NOISE_FLOOR:.5f})")
print(f"test R2        {R2(test, XHatTest):.4f}   (pooled across {N_SERIES} series)")

Slice = slice(0, 600)
Pair = np.stack([test0[Slice], XHatTest0[Slice]])
Fig, Axes = PlotNFeatSeries(
    Pair, Cols=5, Labels=["true", "reconstructed"],
    Title=f"Test reconstruction, series 0, first 600 steps "
          f"($R^2$ = {R2(test0, XHatTest0):.4f}, MSE = {MseNp(test0, XHatTest0):.5f})",
    YLabel="Standardised value")
plt.show()


### Parity

In [ ]:
Fig, Ax = PlotParity(test, XHatTest, Title="Test reconstruction parity",
                     R2=R2(test, XHatTest),
                     XLabel="True observation", YLabel="Reconstructed observation")
plt.show()

Fig, Ax = PlotChannelBar(PerChannelR2(test, XHatTest),
                         Title="Per-channel reconstruction $R^2$ (test)",
                         Baseline=1.0, BaselineLabel="perfect ($R^2 = 1$)")
plt.show()
print(f"worst channel R2 {PerChannelR2(test, XHatTest).min():.4f} "
      f"(channel {int(PerChannelR2(test, XHatTest).argmin())})")

## True vs predicted states

In [ ]:
# The probe absorbs the arbitrary rotation/scale of the latent. Fit on TRAIN
# only, pooled across every training-pool series.
with torch.no_grad():
    _, _, _, BTrain, _ = Best(XTrain)
    _, _, _, BTest, _ = Best(XTest)
BTrainNp, BTestNp = BTrain.cpu().numpy(), BTest.cpu().numpy()

ProbeB = FitStateMap(BTrainNp, StatesTrain)
StatesHat = ApplyStateMap(ProbeB, BTestNp)

# Plotting restricted to series 0: run the same frozen probe on just that
# series' b_t, so the overlay is a single temporally-contiguous trajectory.
with torch.no_grad():
    _, _, _, BTest0, _ = Best(torch.tensor(test0, dtype=torch.float32, device=DEVICE))
StatesHat0 = ApplyStateMap(ProbeB, BTest0.cpu().numpy())

Fig, Axes = PlotNFeatSeries(
    np.stack([StatesTest0[:600], StatesHat0[:600]]), Cols=3,
    Titles=["$x$", "$y$", "$z$"], Labels=["true state", "affine probe of $b_t$"],
    Title="Is the true state linearly decodable from the unbounded latent? "
          "(test, series 0)",
    YLabel="State value")
plt.show()

Fig, Axes = PlotAttractor3D(
    [StatesTest0, StatesHat0], ["True (test, series 0)", "Probed from $b_t$"],
    Width=4.0, Height=4.0,
    Title="Attractor geometry recovered by the affine probe (series 0)")
plt.show()


In [ ]:
Fig, Axes = PlotNFeatSeries(
    BTest0.cpu().numpy()[:800][None], Cols=3, Titles=[f"$b_{i}$" for i in range(K)],
    Title="The unbounded forecast latent $b_t$ on test, series 0, first 800 steps",
    YLabel="Latent value (unbounded)")
plt.show()
print(f"|b| range [{BTestNp.min():.2f}, {BTestNp.max():.2f}]  (pooled across "
      f"{N_SERIES} series, unbounded by design)")


# Model B -- the phi sweep

One latent, one knob. `L = (1 - phi) * L_rec + phi * L_fcst`, swept end to end.

The endpoints are degenerate by construction and that is why they are included: at `phi = 0` the
GRU head receives no gradient at all, and at `phi = 1` the decoder is trained only through the
forecast path. They bracket the frontier honestly rather than flattering it.

In [ ]:
def TrainJoint(Phi, Seed=0, Epochs=EPOCHS):
    def Build():
        SeedAll(Seed)
        return JointAEGRU(N, latent=16, width=WidthJoint).to(DEVICE)

    def Train(Model):
        Opt = torch.optim.Adam(Model.parameters(), lr=LR)
        return EpochTrain(lambda: JointEpoch(Model, Opt, Phi),
                          EvalFn=lambda: ValScores(Model),
                          epochs=Epochs, marks=MARKS, name=f"phi {Phi:g} seed {Seed}",
                          verbose=True)

    Tag = f"{Phi:g}".replace(".", "_")
    return TrainOrLoad(CHECKPOINT_DIR / f"joint_phi{Tag}_seed{Seed}.pt",
                       Build, Train, DEVICE)


JointModels, JointHists = {}, {}
Sweep = tqdm(PHIS, desc="phi sweep")
for Phi in Sweep:
    Model, Hist = TrainJoint(Phi)      # seed 0 -- the "primary" model every plot below uses
    JointModels[Phi], JointHists[Phi] = Model, Hist
    Sweep.write(f"phi {Phi:.2f} | {Hist['epochs_done']:3d} epochs | "
                f"val_recon {Hist['val_recon'][-1]:.5f} | "
                f"val_fcst {Hist['val_fcst'][-1]:.5f}")

# Extra seeds, for the phi values a reviewer is most likely to query. Every
# other phi stays at seed 0 only -- its VPT std in the summary table is
# genuinely n=1, not measured stability; the "seeds" column says so explicitly.
EXTRA_SEED_PHIS = [p for p in (0.1, 0.25, 0.5) if p in PHIS]  # QUICK's PHIS may omit some
JointModelsMultiSeed = {}
for Phi in tqdm(EXTRA_SEED_PHIS, desc="phi extra seeds"):
    Seeded = [JointModels[Phi]]                       # seed 0, already trained above
    for Seed in SEEDS[1:]:
        Model, _ = TrainJoint(Phi, Seed=Seed)
        Seeded.append(Model)
    JointModelsMultiSeed[Phi] = Seeded


In [ ]:
# The knob, seen directly: reconstruction and forecast validation losses as phi moves.
Fig, Ax = PlotLossCurves(
    {rf"$\phi$ = {p:g}": JointHists[p]["val_recon"] for p in PHIS},
    Floor=NOISE_FLOOR,
    Title=r"Joint AE+GRU (model B): validation reconstruction as $\phi$ sweeps 0 to 1",
    YLabel="Validation reconstruction MSE")
plt.show()

Fig, Ax = PlotLossCurves(
    {rf"$\phi$ = {p:g}": JointHists[p]["val_fcst"] for p in PHIS},
    Title=r"Joint AE+GRU (model B): validation forecast as $\phi$ sweeps 0 to 1",
    YLabel="Validation forecast MSE")
plt.show()

Knob = pd.DataFrame([{"phi": p, "epochs": JointHists[p]["epochs_done"],
                      "val_recon": JointHists[p]["val_recon"][-1],
                      "val_fcst": JointHists[p]["val_fcst"][-1]} for p in PHIS])
Knob.round(5)

# Model D -- AEGRU with a sigmoid latent

The one architectural feature Ours relies on for boundedness is a sigmoid on the latent the
decoder reads. Model D asks the direct question: is that alone worth anything, or does the
benefit come from *separating* the bounded and unbounded carriers rather than from bounding
one of them?

`JointAEGRUSigmoid` is `JointAEGRU` (Model B) with one change: the shared latent is bounded to
`[0, 1]` at encode time and kept bounded through the rollout by a residual step in logit space --
the same construction `Src/LatentMapping.py`'s `LatentMappingDynamic` uses, so this is not a
weaker version of boundedness, just the same mechanism applied to a single shared latent instead
of two separated ones. Trained at a single `phi = 0.5` -- the balanced setting, same footing as
the PINN's single oracle run rather than a second full sweep.


In [ ]:
def TrainSigmoid(Phi=0.5, Seed=0, Epochs=EPOCHS):
    def Build():
        SeedAll(Seed)
        return JointAEGRUSigmoid(N, latent=16, width=WidthSigmoid).to(DEVICE)

    def Train(Model):
        Opt = torch.optim.Adam(Model.parameters(), lr=LR)
        return EpochTrain(lambda: JointEpoch(Model, Opt, Phi),
                          EvalFn=lambda: ValScores(Model),
                          epochs=Epochs, marks=MARKS, name=f"sigmoid phi={Phi:g}",
                          PlotFn=LossPlot(["recon", "fcst", "val_recon", "val_fcst"],
                                          Floor=NOISE_FLOOR, Title="AEGRU+sigmoid (D)"),
                          plot_every=PLOT_EVERY, verbose=True)

    Tag = f"{Phi:g}".replace(".", "_")
    return TrainOrLoad(CHECKPOINT_DIR / f"sigmoid_phi{Tag}_seed{Seed}.pt", Build, Train, DEVICE)


SigmoidModel, SigmoidHist = TrainSigmoid()   # seed 0 -- the "primary" model every plot below uses
print(f"sigmoid (D) | {SigmoidHist['epochs_done']} epochs | "
      f"val_recon {SigmoidHist['val_recon'][-1]:.5f} | "
      f"val_fcst {SigmoidHist['val_fcst'][-1]:.5f}")

# Extra seeds: this is the key control on the headline claim (does bounding
# alone explain Ours), so it gets the same 3 seeds Ours and the PINN get.
SigmoidModels = [SigmoidModel]
for _Seed in SEEDS[1:]:
    _Model, _ = TrainSigmoid(Seed=_Seed)
    SigmoidModels.append(_Model)


# Model C -- the PINN

An autoencoder whose 3-d latent is forced to obey the true Lorenz equations, so the forecaster
has **no learned parameters**: rollout is RK4 integration of the known field.

Two caveats stated up front. This model is handed the *exact* governing equations, which no
other model here gets — it is an oracle and upper-bounds what physics-decoupling can buy. And
because `lift()` z-scores the states before lifting, the recoverable latent is standardised
Lorenz, not physical Lorenz; the learnable affine carries that gauge and is seeded from the true
state statistics. Nothing is supervised on `states` — only the affine *initialisation* uses it.

The misspecified run (rho = 26 instead of 28) is the honest control: it asks how much of the
advantage survives when the physics is merely approximately right, which is the only situation
that ever obtains outside a synthetic testbed.

In [ ]:
T3Np = MakeSeriesWindows(TrainSeries, length=3, stride=1)
T3 = torch.tensor(T3Np, dtype=torch.float32, device=DEVICE)
print(f"physics triples {tuple(T3.shape)}  (boundary-safe across {N_SERIES} series)")


def TrainPinn(Seed, Epochs=EPOCHS, Rho=28.0, Verbose=True):
    def Build():
        SeedAll(Seed)
        Model = PhysicsLatentAE(N, width=WidthPinn, dt=DT, rho=Rho).to(DEVICE)
        Model.StatsInit(StatesTrain)   # mu/logsd are learnable, so a loaded
        return Model                   # checkpoint overwrites this init anyway

    def Train(Model):
        Opt = torch.optim.Adam(Model.parameters(), lr=LR)

        def Epoch():
            # Task 4 fix: recalibrate the physics coordinate gauge by least
            # squares against the true state (training data only), BEFORE
            # each epoch's gradient steps. Without this, mu/logsd drift under
            # a purely relative physics residual and s = ToPhysical(zhat)
            # lands off the true attractor -- see PhysicsLatentAE.CalibrateGauge
            # and the diagnostic cells above for the full argument.
            Model.CalibrateGauge(XTrain, StatesTrain)
            return PinnEpoch(Model, Opt, T3)

        Hist = EpochTrain(Epoch, EvalFn=lambda: ValScores(Model),
                          epochs=Epochs, marks=MARKS, name=f"pinn rho={Rho:g} (seed {Seed})",
                          PlotFn=LossPlot(["recon", "physics", "val_recon", "val_fcst"],
                                          Floor=NOISE_FLOOR,
                                          Title=f"PINN, rho={Rho:g}") if Verbose else None,
                          plot_every=PLOT_EVERY, verbose=Verbose)
        Model.CalibrateGauge(XTrain, StatesTrain)   # final calibration, for eval/rollout
        return Hist

    Tag = f"{Rho:g}".replace(".", "_")
    return TrainOrLoad(CHECKPOINT_DIR / f"pinn_rho{Tag}_seed{Seed}.pt", Build, Train, DEVICE)


PinnModels, PinnHists = [], []
for Seed in SEEDS:
    Model, Hist = TrainPinn(Seed, Verbose=(Seed == SEEDS[0]))
    PinnModels.append(Model); PinnHists.append(Hist)

PinnBadModel, PinnBadHist = TrainPinn(SEEDS[0], Rho=26.0, Verbose=False)


### Task 4 diagnostics: was the PINN's forecast actually broken, and why

Symptom this section exists to explain: `val_recon` and `physics` both looked healthy while
`val_fcst` rose across training. The four checks below are the ones that distinguish "encoder
representation is fine, only integration is broken" from "encoder never learned the state" --
run against the trained, now-calibrated model, with the pre-fix numbers alongside for comparison.


In [ ]:
PinnRef = PinnModels[0]

# 1. Latent scale vs true Lorenz scale -- the calibrated gauge, applied to
# training data, should land close to the true attractor's range, not a
# standardised [-3, 3]-ish box.
with torch.no_grad():
    ZhatTrain = PinnRef.Encode(XTrain.unsqueeze(1)).squeeze(1)
    STrain = PinnRef.ToPhysical(ZhatTrain).cpu().numpy()
print("1. latent scale (calibrated s = ToPhysical(zhat)) vs true Lorenz state, on TRAIN:")
print(f"   s     min {STrain.min(0).round(2)}  max {STrain.max(0).round(2)}  "
      f"mean {STrain.mean(0).round(2)}  std {STrain.std(0).round(2)}")
print(f"   true  min {StatesTrain.min(0).round(2)}  max {StatesTrain.max(0).round(2)}  "
      f"mean {StatesTrain.mean(0).round(2)}  std {StatesTrain.std(0).round(2)}")

# 2. Linear probe from the PINN's calibrated s -> true state, R2 per dimension.
# The gauge IS a least-squares fit to the true state, so this R2 is close to
# what CalibrateGauge itself optimised -- it is reported here as the
# representation-quality number, the same quantity the b_t -> state probe
# reports for "Ours".

# 3. Integration dt: must match the data's generation dt (0.01).
print(f"\n2. integration dt = {PinnRef.dt}   data generation dt = {DT}   match: {PinnRef.dt == DT}")

# 4. Train/eval rollout parity: the model has no fixed horizon of its own --
# confirm the eval horizon (HORIZON, used by Warm/Fut) is what Results below
# actually evaluates against, and that it is not longer than what val_fcst
# was scored on during training (UNROLL).
print(f"\n3. training rollout depth (UNROLL) = {UNROLL} steps   "
      f"eval rollout horizon (HORIZON) = {HORIZON} steps")
print("   eval horizon is longer than the training rollout by construction -- VPT/NRMSE@5LT")
print("   measure genuine free-running extrapolation, not a length mismatch bug.")

print(f"\nfinal: val_recon {PinnHists[0]['val_recon'][-1]:.5f}  "
      f"physics {PinnHists[0]['physics'][-1]:.4f}  "
      f"val_fcst {PinnHists[0]['val_fcst'][-1]:.4f}  (pre-fix: 0.0037 / 0.031 / 17.6-ish and rising)")


In [ ]:
Fig, Ax = PlotLossCurves(
    {k: PinnHists[0][k] for k in ("recon", "physics", "val_recon", "val_fcst")},
    Floor=NOISE_FLOOR,
    Title=r"PINN (model C, true physics $\rho$ = 28): reconstruction and physics residual")
plt.show()

Fig, Ax = PlotLossCurves(
    {r"$\rho$ = 28 (true), val_recon": PinnHists[0]["val_recon"],
     r"$\rho$ = 26 (misspecified), val_recon": PinnBadHist["val_recon"],
     r"$\rho$ = 28 (true), physics": PinnHists[0]["physics"],
     r"$\rho$ = 26 (misspecified), physics": PinnBadHist["physics"]},
    Title="PINN: true against misspecified governing equations")
plt.show()

print(f"pinn rho=28 | {PinnHists[0]['epochs_done']} epochs | "
      f"val_recon {PinnHists[0]['val_recon'][-1]:.5f} | "
      f"physics {PinnHists[0]['physics'][-1]:.4f}")
print(f"pinn rho=26 | {PinnBadHist['epochs_done']} epochs | "
      f"val_recon {PinnBadHist['val_recon'][-1]:.5f} | "
      f"physics {PinnBadHist['physics'][-1]:.4f}")

# Evaluation

In [ ]:
EvalKw = dict(dt=DT, threshold=THRESHOLD, scale=SCALE,
              noise_floor=NOISE_FLOOR, bounds=BOUNDS)


def Agg(Models, Label, **Kw):
    """Evaluate every seed of a model and average, keeping the spread."""
    Runs = [EvaluateModel(m, Warm, Fut, **EvalKw, **Kw) for m in Models]
    Out = {"model": Label, "n_seeds": len(Runs)}
    for k in Runs[0]:
        if k == "curve":
            Out["curve"] = np.mean([r["curve"] for r in Runs], axis=0)
        elif isinstance(Runs[0][k], (int, float, bool)):
            Vals = [float(r[k]) for r in Runs]
            Out[k] = float(np.mean(Vals))
            Out[k + "_std"] = float(np.std(Vals))
    return Out


Results = {}
Bar = tqdm(total=4 + len(PHIS), desc="evaluating")
Results["Ours (two-stage)"] = Agg(OursModels, "Ours (two-stage)"); Bar.update()
Results["PINN (true physics)"] = Agg(PinnModels, "PINN (true physics)"); Bar.update()
Results["PINN (rho=26)"] = Agg([PinnBadModel], "PINN (rho=26)"); Bar.update()
Results["AEGRU+sigmoid (D)"] = Agg(SigmoidModels, "AEGRU+sigmoid (D)"); Bar.update()  # noqa (multi-seed)
for Phi in PHIS:
    Models = JointModelsMultiSeed.get(Phi, [JointModels[Phi]])
    Results[f"phi={Phi:g}"] = Agg(Models, f"phi={Phi:g}"); Bar.update()
Bar.close()

# Trivial references. Anything that cannot beat these is broken, not interesting.
for Name, Pred in [("persistence", PersistenceRollout(WarmNp, HORIZON)),
                   ("climatology", MeanRollout(WarmNp, HORIZON, train.mean(0)))]:
    Fm = ForecastMetrics(FutNp, Pred, DT, THRESHOLD, SCALE, BOUNDS)
    Results[Name] = {"model": Name, "recon_nrmse": np.nan, "recon_mse": np.nan,
                     "recon_r2": np.nan, **Fm}

print(f"evaluated {len(Results)} configurations on {Warm.shape[0]} held-out windows")

# Epoch counts, seed-0 (the "primary" run) per model -- for the final summary
# table's seeds/epochs columns. Ours reports stageA + stageB combined, since
# it is the only two-stage schedule here. Every model gets the same EPOCHS
# passes over its own data now, so this column is mostly a sanity check.
EpochsByModel = {
    "Ours (two-stage)": OursHists[0]["stageA"]["epochs_done"] + OursHists[0]["stageB"]["epochs_done"],
    "PINN (true physics)": PinnHists[0]["epochs_done"],
    "PINN (rho=26)": PinnBadHist["epochs_done"],
    "AEGRU+sigmoid (D)": SigmoidHist["epochs_done"],
}
for Phi in PHIS:
    EpochsByModel[f"phi={Phi:g}"] = JointHists[Phi]["epochs_done"]

# --- Compute-scaling check, from the MARKS snapshots instead of a separate 10x
# run: for every model, has val_fcst plateaued by the final mark, or is it
# still moving? A model still moving a lot between the last two marks is
# undertrained at EPOCHS, not frontier-limited; one that has flattened out is
# the opposite. This replaces the old single 1x-vs-10x comparison with the
# whole trajectory, read off the runs already trained above -- no extra models.
def _MarkSeries(Hist, Key):
    return [(e, Hist["marks"][e][Key]) for e in sorted(Hist["marks"]) if Key in Hist["marks"][e]]


print("\ncompute-scaling check (val_fcst at each mark, seed 0):")
_LadderRows = []
_LadderHists = [("Ours (two-stage)", OursHists[0]["stageB"]),
                ("PINN (true physics)", PinnHists[0]),
                ("AEGRU+sigmoid (D)", SigmoidHist)] + [(f"phi={p:g}", JointHists[p]) for p in PHIS]
for Label, Hist in _LadderHists:
    Series = _MarkSeries(Hist, "val_fcst")
    if len(Series) < 2:
        continue
    (e0, v0), (e1, v1) = Series[-2], Series[-1]
    RelMove = abs(v1 - v0) / max(abs(v0), 1e-12)
    Status = "still moving" if RelMove > 0.02 else "plateaued"
    print(f"  {Label:22s} epoch {e0:3d}->{e1:3d}  val_fcst {v0:.4f}->{v1:.4f}  "
         f"({RelMove:+.1%})  {Status}")
    for e, v in Series:
        _LadderRows.append({"model": Label, "epoch": e, "val_fcst": v})

EpochLadder = pd.DataFrame(_LadderRows)


## Reconstruction

In [ ]:
Rec = pd.DataFrame([{"model": r["model"],
                     "recon MSE": r.get("recon_mse"),
                     "x noise floor": r.get("recon_mse_over_floor"),
                     "recon NRMSE": r.get("recon_nrmse"),
                     "R2": r.get("recon_r2"),
                     "worst channel R2": r.get("recon_r2_worst")}
                    for r in Results.values()
                    if not np.isnan(r.get("recon_mse", np.nan))])
Rec = Rec.sort_values("recon MSE").reset_index(drop=True)

Fig, Ax = plt.subplots(figsize=(6.8, 3.2), dpi=DPI)
Ax.bar(np.arange(len(Rec)), Rec["recon MSE"], width=0.75, color=PALETTE[0],
       label="test reconstruction MSE")
Ax.axhline(NOISE_FLOOR, color="0.25", linestyle="--", linewidth=0.9,
           label=f"noise floor ({NOISE_FLOOR:.5f})")
Ax.set_xticks(np.arange(len(Rec)))
Ax.set_xticklabels(Rec["model"], rotation=45, ha="right", fontsize=6)
Ax.set_yscale("log")
Ax.set_xlabel("Model (matched params, matched wall clock)")
Ax.set_ylabel("Reconstruction MSE, log scale  (lower is better)")
Ax.set_title("Reconstruction on held-out windows")
Ax.grid(True, axis="y", linewidth=0.4, alpha=0.4)
Ax.set_axisbelow(True)
Ax.legend(loc="upper left")
plt.show()

Rec.round(5)

### Why does `phi=0` score below 1.0x the noise floor?

This persists after both the noise-floor fix and the standardisation-leak fix (Tasks 1-2), which
rules out both as the explanation -- a genuine data or eval-pipeline leak would move every model's
score, not one. The leading hypothesis: `JointAEGRU`'s shared latent is 16-dimensional
(`Comp/JointAEGRU.py`) against a true intrinsic dimensionality of 3, and `phi=0` is the only
setting where no forecast term ever regularises that latent toward simplicity -- free to spend all
16 dimensions purely on reconstruction, it can exploit redundancy across 30 correlated,
independently-noised channels to denoise past the naive per-channel floor. Tested directly below:
retrain the identical `phi=0` configuration with `latent=3`, matching the true dimensionality and
the capacity "Ours" uses, and see whether the effect disappears.


In [ ]:
# Diagnostic control, not part of the model comparison: same architecture and
# training as phi=0 above, only latent changed from 16 (JointAEGRU's default,
# width-matched to the 124,482-param budget) to 3 (the true Lorenz dimension,
# matching what "Ours" uses). If beating the floor was spare capacity, not a
# leak, this should land at or above 1.0x.
WidthJoint3 = MatchWidth(lambda w: JointAEGRU(N, latent=3, width=w), TARGET)


def BuildDiagLatent3():
    SeedAll(0)
    return JointAEGRU(N, latent=3, width=WidthJoint3).to(DEVICE)


def TrainDiagLatent3(Model):
    Opt = torch.optim.Adam(Model.parameters(), lr=LR)
    return EpochTrain(lambda: JointEpoch(Model, Opt, 0.0),
                      EvalFn=lambda: ValScores(Model),
                      epochs=EPOCHS, marks=MARKS, name="diag latent=3 phi=0")


DiagModel3, DiagHist3 = TrainOrLoad(CHECKPOINT_DIR / "diag_latent3_phi0_seed0.pt",
                                    BuildDiagLatent3, TrainDiagLatent3, DEVICE)
DiagResult3 = EvaluateModel(DiagModel3, Warm, Fut, **EvalKw)

print(f"phi=0, latent=16 (the actual Model B row above): "
      f"recon MSE {Results['phi=0']['recon_mse']:.5f}  "
      f"x floor {Results['phi=0']['recon_mse_over_floor']:.5f}")
print(f"phi=0, latent=3  (this diagnostic, capacity-matched): "
      f"recon MSE {DiagResult3['recon_mse']:.5f}  "
      f"x floor {DiagResult3['recon_mse_over_floor']:.5f}")
if DiagResult3["recon_mse_over_floor"] < 1.0:
    print("STILL below the floor with latent=3 -- capacity is NOT the explanation, "
         "something else is; treat the sub-floor result as unresolved.")
else:
    print("At or above the floor with latent=3 -- CONFIRMS spare latent capacity (16 vs "
         "the true 3) explains phi=0 beating the floor at latent=16. Not a leak.")


## Forecast

In [ ]:
Curves = {k: r["curve"] for k, r in Results.items()}
Fig, Ax = PlotHorizonCurves(
    Curves, Dt=DT, Lam=LAMBDA_MAX, Threshold=THRESHOLD,
    Title=f"Free-running forecast error over {HORIZON / STEPS_PER_LYAP:.0f} Lyapunov "
          f"times, {Warm.shape[0]} independent windows")
plt.show()

In [ ]:
Fc = pd.DataFrame([{"model": r["model"],
                    "VPT (Lyap)": r.get("vpt"),
                    "VPT std": r.get("vpt_std", 0.0),
                    "censored": bool(r.get("vpt_censored") or False),
                    "NRMSE @ H": r.get("fcst_nrmse_full"),
                    "divergence": r.get("divergence")}
                   for r in Results.values()])
Fc = Fc.sort_values("VPT (Lyap)", ascending=False).reset_index(drop=True)

Fig, Ax = plt.subplots(figsize=(6.8, 3.2), dpi=DPI)
Ax.bar(np.arange(len(Fc)), Fc["VPT (Lyap)"], yerr=Fc["VPT std"], width=0.75,
       capsize=2, color=PALETTE[0], ecolor="0.25",
       label=f"VPT at NRMSE {THRESHOLD:g}  (error bars: std over seeds)")
Ax.set_xticks(np.arange(len(Fc)))
Ax.set_xticklabels(Fc["model"], rotation=45, ha="right", fontsize=6)
Ax.set_xlabel("Model (matched params, matched wall clock)")
Ax.set_ylabel("Valid prediction time, Lyapunov times  (higher is better)")
Ax.set_title(f"Valid prediction time  ({STEPS_PER_LYAP:.0f} steps = 1 Lyapunov time)")
Ax.grid(True, axis="y", linewidth=0.4, alpha=0.4)
Ax.set_axisbelow(True)
Ax.legend(loc="upper right")
plt.show()

Fc.round(4)

`censored = True` means the error curve never crossed the threshold inside the horizon, so the
VPT is a lower bound rather than a measurement — read those rows as "at least this".

### What the rollouts actually look like

In [ ]:
Show = {"Ours (two-stage)": OursModels[0], "PINN (true physics)": PinnModels[0],
        f"phi={PHIS[len(PHIS) // 2]:g}": JointModels[PHIS[len(PHIS) // 2]]}
Win = 0
with torch.no_grad():
    Rolls = {k: m.Rollout(Warm[Win:Win + 1], HORIZON)[0].cpu().numpy()
             for k, m in Show.items()}

for Name, Pred in Rolls.items():
    Fig, Axes = PlotNFeatSeries(
        np.stack([FutNp[Win, :, :10], Pred[:, :10]]), Cols=5,
        Labels=["true", "free-running forecast"],
        Title=f"{Name}: forecast against truth, first 10 channels, "
              f"{HORIZON / STEPS_PER_LYAP:.0f} Lyapunov times",
        YLabel="Standardised value")
    plt.show()

In [ ]:
# Same rollouts pushed through the observation -> state probe, so the divergence
# is visible in physical coordinates. Fit on the pooled training pool.
ProbeW = FitStateMap(train, StatesTrain)

TrueState = ApplyStateMap(ProbeW, FutNp[Win])
Fig, Axes = PlotNFeatSeries(
    np.stack([TrueState] + [ApplyStateMap(ProbeW, p) for p in Rolls.values()]),
    Cols=3, Titles=["$x$", "$y$", "$z$"], Labels=["truth"] + list(Rolls),
    Title="Forecast in probed state space  (observations pushed through the "
          "obs -> state affine probe)",
    YLabel="State value")
plt.show()


## Bounded latent through the rollout

`m` ends in a sigmoid, so `C` is inside `[0, 1]` at every step of an arbitrarily long rollout
*by construction* -- however far `b` drifts. The violation count is therefore a correctness check
rather than a result: a non-zero value is a bug. What is a result is whether the decoded forecast
stays inside the range the model was trained on, which no baseline here guarantees, and how much
of `C` is pinned at the rails while it does.


In [ ]:
with torch.no_grad():
    XRoll, BRoll, CRoll = OursModels[0].RolloutLatents(WarmC, CLIMATE)
CRollNp, BRollNp = CRoll.cpu().numpy(), BRoll.cpu().numpy()

assert BoundViolation(CRollNp) == 0.0, "bounded latent escaped [0,1] -- this is a bug"
print(f"C outside [0,1] over {CLIMATE} steps x {CRollNp.shape[0]} rollouts: "
      f"{BoundViolation(CRollNp):.6f}")
print(f"C saturated (outside [0.02, 0.98]): {Saturation(CRollNp):.4f}")
print(f"|b| max during rollout: {np.abs(BRollNp).max():.2f}")

Steps = [0, 10, 50, 100, 250, 500, 1000, 2000, 3500, 5000, CLIMATE - 1]
Fig, Axes = PlotMatrixGrid(
    [CRollNp[0, s] for s in Steps], Titles=[f"$t+{s}$" for s in Steps], Cols=6,
    Title=f"Bounded latent $C_t$ ({A}x{B}) through a "
          f"{CLIMATE / STEPS_PER_LYAP:.0f}-Lyapunov-time rollout",
    CbarLabel="$C$ entry value (bounded to $[0, 1]$)")
plt.show()

In [ ]:
Fig, Ax = plt.subplots(1, 2, figsize=(8.4, 3.0), dpi=DPI)

Ax[0].hist(CRollNp.ravel(), bins=80, color=PALETTE[0],
           label=f"all $C$ entries ({CRollNp.size:,})")
Ax[0].axvline(0.02, color="0.25", linestyle="--", linewidth=0.9,
              label="saturation band [0.02, 0.98]")
Ax[0].axvline(0.98, color="0.25", linestyle="--", linewidth=0.9)
Ax[0].set_title("Distribution of bounded-latent entries over the rollout")
Ax[0].set_xlabel("$C$ entry value")
Ax[0].set_ylabel("Count")
Ax[0].legend(loc="upper center")

Ax[1].plot(np.abs(BRollNp[0]).max(axis=1), linewidth=0.9, color=PALETTE[1],
           label=r"$\max_i |b_i|$, rollout 0")
Ax[1].set_yscale("log")
Ax[1].set_title("Unbounded latent magnitude during the same rollout")
Ax[1].set_xlabel("Rollout step")
Ax[1].set_ylabel(r"$\max_i |b_i|$, log scale")
Ax[1].legend(loc="upper left")

for A_ in Ax:
    A_.grid(True, linewidth=0.4, alpha=0.4)
    A_.set_axisbelow(True)
Fig.suptitle("Boundedness audit: $C$ stays in $[0, 1]$ however far $b$ drifts")
plt.show()

## Climate: does the long rollout stay on the attractor?

Short-horizon error says nothing about this. A model can track for two Lyapunov times and then
settle onto a fixed point or a limit cycle, and its NRMSE curve would look no different from one
that keeps orbiting correctly. These statistics run to 50 Lyapunov times, long past the point
where pointwise error has saturated.

In [ ]:
@torch.no_grad()
def ClimateOf(Model, **Kw):
    Model.eval()
    Pred = Model.Rollout(WarmC, CLIMATE, **Kw).cpu().numpy()
    St = AttractorStats(FutCNp, Pred)
    St["divergence"] = float(((Pred < BOUNDS[0]) | (Pred > BOUNDS[1])).any(axis=(1, 2)).mean())
    St["final_nrmse"] = float(NRMSE(FutCNp[:, -500:], Pred[:, -500:], SCALE))
    return St, Pred


ClimateModels = {"Ours (two-stage)": OursModels[0],
                 "PINN (true physics)": PinnModels[0],
                 "PINN (rho=26)": PinnBadModel,
                 "AEGRU+sigmoid (D)": SigmoidModel}
ClimateModels.update({f"phi={p:g}": JointModels[p]
                      for p in (PHIS[0], PHIS[len(PHIS) // 2], PHIS[-1])})

Climate, ClimatePreds = {}, {}
for Name, Model in tqdm(ClimateModels.items(), desc="climate"):
    Climate[Name], ClimatePreds[Name] = ClimateOf(Model)

pd.DataFrame([{"model": k, **v} for k, v in Climate.items()]).round(4)

In [ ]:
# The Lorenz map: successive maxima of z, recovered through the probe. The true
# system traces a thin tent; a model that has the climate but not the geometry
# produces a visibly fattened or collapsed one.
TrueMap = ZMaxMap(ApplyStateMap(ProbeW, FutCNp[0])[:, 2])
Names = list(ClimatePreds)

Fig, Axes = plt.subplots(1, len(Names) + 1, figsize=(2.4 * (len(Names) + 1), 2.8),
                         dpi=DPI, sharex=True, sharey=True)
Axes[0].scatter(TrueMap[:, 0], TrueMap[:, 1], s=1.5, color="0.25")
Axes[0].set_title(f"truth\n({len(TrueMap)} maxima)", fontsize=7.5)
for Ax, Name, Col in zip(Axes[1:], Names, PALETTE):
    M = ZMaxMap(ApplyStateMap(ProbeW, ClimatePreds[Name][0])[:, 2])
    if len(M):
        Ax.scatter(M[:, 0], M[:, 1], s=1.5, color=Col)
    Ax.set_title(f"{Name}\n({len(M)} maxima)", fontsize=7.5)
for Ax in Axes:
    Ax.set_xlabel("$z_n$", fontsize=7.5)
    Ax.grid(True, linewidth=0.3, alpha=0.4)
    Ax.set_axisbelow(True)
Axes[0].set_ylabel("$z_{n+1}$", fontsize=7.5)
Fig.suptitle(f"Lorenz return map after {CLIMATE / STEPS_PER_LYAP:.0f} Lyapunov "
             f"times  (the true system traces a thin tent)")
plt.show()

In [ ]:
# The attractor itself, in probed state space.
Fig, Axes = PlotAttractor3D(
    [ApplyStateMap(ProbeW, FutCNp[0])] + [ApplyStateMap(ProbeW, ClimatePreds[n][0]) for n in Names],
    ["truth"] + Names, Cols=4, Width=3.0, Height=3.0,
    Title=f"Attractor after {CLIMATE / STEPS_PER_LYAP:.0f} Lyapunov times, "
          f"in probed state space")
plt.show()

In [ ]:
# Marginal distributions -- the invariant measure the Wasserstein column scores.
Fig, Axes = plt.subplots(1, 3, figsize=(9.6, 3.0), dpi=DPI)
TrueS = ApplyStateMap(ProbeW, FutCNp.reshape(-1, N))
for i, Lab in enumerate(["$x$", "$y$", "$z$"]):
    Axes[i].hist(TrueS[:, i], bins=70, density=True, histtype="step",
                 linewidth=1.6, color="0.25", label="truth")
    for j, Name in enumerate(Names):
        S = ApplyStateMap(ProbeW, ClimatePreds[Name].reshape(-1, N))
        Axes[i].hist(S[:, i], bins=70, density=True, histtype="step",
                     linewidth=1.0, color=PALETTE[j % len(PALETTE)], label=Name)
    Axes[i].set_title(f"Coordinate {Lab}")
    Axes[i].set_xlabel(f"State coordinate {Lab}")
    Axes[i].grid(True, linewidth=0.4, alpha=0.4)
    Axes[i].set_axisbelow(True)
Axes[0].set_ylabel("Probability density")
Fig.legend(*Axes[0].get_legend_handles_labels(), loc="outside lower center",
           ncol=min(len(Names) + 1, 4), fontsize=6.5)
Fig.suptitle(f"Invariant measure per state coordinate, "
             f"{CLIMATE / STEPS_PER_LYAP:.0f} Lyapunov times")
plt.show()

## Speed under the matched budget

In [ ]:
Timing = []
for Name, Model in [("Ours (two-stage)", OursModels[0]),
                    ("PINN (true physics)", PinnModels[0]),
                    ("AEGRU+sigmoid (D)", SigmoidModel),
                    (f"phi={PHIS[len(PHIS) // 2]:g}", JointModels[PHIS[len(PHIS) // 2]])]:
    T = TimeRollout(Model, Warm, HORIZON, repeats=3)
    Timing.append({"model": Name,
                   "trainable": CountParams(Model),
                   "inference params": CountInferenceParams(Model),
                   "FLOPs/fwd": CountFlops(Model),
                   f"rollout s ({Warm.shape[0]} x {HORIZON})": round(T["rollout_s"], 3),
                   "ms/step": round(T["ms_per_step"], 3)})
pd.DataFrame(Timing)

In [ ]:
Hists = {"Ours (two-stage)": OursHists[0], "PINN (true physics)": PinnHists[0]}
Hists.update({f"phi={p:g}": JointHists[p] for p in PHIS})
Fig, Ax = PlotTimeToQuality(
    Hists, Key="val_recon", Floor=NOISE_FLOOR,
    YLabel="Validation reconstruction MSE  (lower is better)",
    Title=f"Time to quality: every model gets {EPOCHS} epochs over its own data")
plt.show()

print("time to reach 2x the noise floor:")
for k, h in Hists.items():
    t = TimeToTarget(h, "val_recon", 2 * NOISE_FLOOR)
    print(f"  {k:24s} {('%.1fs' % t) if t else 'not reached'}")

# Headline: the tradeoff as a curve

The grey line is the phi sweep — the empirical Pareto front of the balancing approach. Every
other model is a point placed against it.

- A point **above and left** of the curve reconstructs *and* forecasts better than any setting of
  the knob achieves. That is dissolution.
- A point **on** the curve is another way of choosing the same tradeoff. That is balancing under
  a different name, and it means the tension is real.

In [ ]:
Frontier = [(f"{p:g}", Results[f"phi={p:g}"]) for p in PHIS]
Points = {k: Results[k] for k in
         ("Ours (two-stage)", "PINN (true physics)", "PINN (rho=26)",
          "AEGRU+sigmoid (D)")}

FloorNrmse = float(np.sqrt(NOISE_FLOOR) / SCALE)
Fig, Ax = PlotParetoFront(
    Frontier, Points, NoiseFloor=FloorNrmse,
    Title="Reconstruction against forecast horizon\n"
          f"matched parameters ({TARGET:,}) and matched epochs ({EPOCHS})")
plt.show()


## Ablations

**Does the two-stage schedule matter?** Train the same architecture end-to-end, forecast objective
included, so the forecast loss reshapes `b` and therefore reconstruction. If joint training lands
on the phi curve while two-stage does not, the decoupling is doing the work rather than the
architecture.


In [ ]:
def TrainOursJoint(Seed, Epochs=EPOCHS):
    """Same modules, one stage, forecast objective included from the start."""
    def Build():
        SeedAll(Seed)
        return DecoupledModel(N, K, A, B).to(DEVICE)

    def Train(Model):
        return _TrainOursJointBody(Model, Epochs)

    return TrainOrLoad(CHECKPOINT_DIR / f"oursjoint_seed{Seed}.pt", Build, Train, DEVICE)


def _TrainOursJointBody(Model, Epochs):
    Opt = torch.optim.Adam(Model.parameters(), lr=LR)

    def Epoch():
        Perm = torch.randperm(XFull.shape[0])
        Sums, Nb = [0.0, 0.0], 0
        Batches = tqdm(range(0, len(Perm), BATCH_SEQ), desc="ours joint", leave=False)
        for i in Batches:
            Seq = XFull[Perm[i:i + BATCH_SEQ]]
            Flat = Seq.reshape(-1, N)
            XHat, Ct, CTeach, _, _ = Model(Flat)
            Rec = MseT(XHat, Flat) + W_CONSIST * MseT(Ct, CTeach)
            Fc = MseT(Model.Rollout(Seq[:, :WARM], UNROLL), Seq[:, WARM:])
            Loss = 0.5 * Rec + 0.5 * Fc
            Opt.zero_grad(); Loss.backward(); Opt.step()
            Sums = [Sums[0] + Rec.item(), Sums[1] + Fc.item()]
            Nb += 1
            Batches.set_postfix(recon=f"{Rec.item():.2e}", fcst=f"{Fc.item():.2e}")
        Nb = max(Nb, 1)
        return {"recon": Sums[0] / Nb, "fcst": Sums[1] / Nb}

    return EpochTrain(Epoch, EvalFn=lambda: ValScores(Model), epochs=Epochs, marks=MARKS,
                      name="ours joint",
                      PlotFn=LossPlot(["recon", "fcst", "val_recon", "val_fcst"],
                                      Floor=NOISE_FLOOR, Title="Ours, joint one-stage"),
                      plot_every=max(PLOT_EVERY // 2, 1))


OursJointModel, OursJointHist = TrainOursJoint(SEEDS[0])


In [ ]:
Ablation = {"Ours (two-stage)": Results["Ours (two-stage)"],
            "Ours (joint, one stage)": Agg([OursJointModel], "Ours (joint, one stage)")}

Fig, Ax = PlotHorizonCurves(
    {k: v["curve"] for k, v in Ablation.items()},
    Dt=DT, Lam=LAMBDA_MAX, Threshold=THRESHOLD,
    Title="Ablation: two-stage against joint one-stage training, forecast error")
plt.show()

Fig, Ax = PlotParetoFront(
    Frontier, Ablation, NoiseFloor=FloorNrmse,
    Title="Ablation: two-stage against joint one-stage,\nplaced on the same frontier")
plt.show()

pd.DataFrame([{"model": r["model"], "recon MSE": r["recon_mse"],
               "recon NRMSE": r["recon_nrmse"], "VPT (Lyap)": r["vpt"],
               "NRMSE @ H": r["fcst_nrmse_full"]}
              for r in Ablation.values()]).round(4)

# Summary

In [ ]:
Summary = pd.DataFrame([{
    "model": r["model"],
    "seeds": r.get("n_seeds", np.nan),
    "epochs": EpochsByModel.get(r["model"], np.nan),
    "recon MSE": r.get("recon_mse"),
    "recon NRMSE": r.get("recon_nrmse"),
    "VPT (Lyap)": r.get("vpt"),
    "VPT std": r.get("vpt_std", 0.0),
    "NRMSE @ 5LT": r.get("fcst_nrmse_full"),
    "divergence": r.get("divergence"),
} for r in Results.values()]).round(4)

print(f"Every trained row: {TARGET:,} trainable params, {EPOCHS} epochs over its own data, "
      f"{THREADS} CPU threads. seeds/epochs columns above make explicit which rows are "
      f"1-seed points (persistence/climatology have neither -- not trained at all).")
Summary.sort_values("VPT (Lyap)", ascending=False).reset_index(drop=True)


## Reading this

The result is whatever the Pareto plot shows, and both outcomes are publishable:

- **Ours above the frontier** — the tradeoff observed across the reconstruction+prediction
  literature is an artifact of forcing one latent to serve both objectives, and separating the
  carriers escapes it on Lorenz. The next step is the cross-domain claim: turbofan, weather,
  bearing.
- **Ours on the frontier** — the tension is structural, at least at this capacity, and the
  interesting question becomes *why*: whether it is the rate-distortion frontier itself, or a
  bound specific to fixed-capacity bounded latents that adaptive precision could still escape.

What this notebook does **not** establish.

- **One system, one lift, one noise level.** The phenomenon claim needs turbofan, weather and
  bearing before it is a claim about representations rather than about Lorenz.
- **No published latent-forecast baseline.** The weighted-loss AE+GRU is the balancing approach
  in its plainest form, not a named method, so a reader can fairly call it a construction. A
  Consistent Koopman Autoencoder (Azencot et al., ICML 2020) is the cheapest fix: encoder/decoder
  plus a linear operator with forward/backward consistency, ~100 lines, and it reports both
  metrics natively. Worth noting that Koopa (NeurIPS 2023) explicitly *drops* the reconstruction
  loss to get its forecasting numbers — a SOTA paper conceding the tradeoff in its own abstract.
- **The PINN is an oracle**, handed the exact governing equations. The rho=26 row is the only
  honest read on what physics buys when it is merely approximately right.
- **Equal epoch split is untuned.** Both stages now get the same EPOCHS
  passes, untuned -- Stage A reconstructs on a much larger per-timestep pool per
  pass than Stage B sees of its own b-space windows, so an equal split is a
  convenient default, not a searched one. Whether a different split moves the
  frontier is untested.

# Noise robustness sweep

Everything above runs at the data's one noise level (5% of each channel's std). This section
re-runs the headline comparison -- Ours, `phi=0.1`, `phi=0.5`, `AEGRU+sigmoid (D)`, persistence,
climatology -- at two more noise levels, 15% and 30%, to see whether the gap between Ours and the
knob's best setting widens, narrows, or holds as the data gets noisier.

Same architecture, same EPOCHS-per-model protocol, same 8-series-pool-plus-32-holdout
construction as the main comparison (including the Task 2 leak fix) -- only `noise` changes. Single seed per model
at each new noise level, not three: this is six extra models trained twice over, and the question
here is a qualitative direction (does the gap move, and which way), not a precision estimate. The
existing noise=0.05 row is the already-seeded result from above, reused rather than re-run.


In [ ]:
from SyntheticGenerators.LorenzLift import save_multi_series_dataset

NOISE_SWEEP_LEVELS = [0.15, 0.30]
_BASE_NOISE_KW = dict(n_series=8, n_steps=20000, n_obs=30, dt=0.01, seed=0,
                      train_frac=0.7, val_frac=0.15,
                      n_holdout=32, holdout_steps=12000, holdout_burn_in=2000)


def RunNoiseLevel(Noise, Seed=0):
    """Train+evaluate the headline models on a fresh dataset at a different
    observation noise level, reusing the main comparison's own training
    functions by temporarily repointing the globals they read.
    """
    global XTrain, Loader, TrainSeries, XFull, XValFull, XValWarm, XValFut, NOISE_FLOOR, CHECKPOINT_DIR
    NoisePath = ROOT / "Data" / f"LorenzLiftMulti_noise{Noise:g}.npz"
    if not NoisePath.exists():
        save_multi_series_dataset(path=str(NoisePath), noise=Noise, **_BASE_NOISE_KW)

    NM = LoadLorenzMultiSeries(NoisePath)
    NTrainSeries, NValSeries = NM["train"], NM["val"]
    NHoldoutObs = NM["holdout_obs"]
    NNoiseFloor = float(NM["noise_floor_mse"])
    NTrainFlat = NTrainSeries.reshape(-1, NTrainSeries.shape[-1])

    NXTrain = torch.tensor(NTrainFlat, dtype=torch.float32, device=DEVICE)
    NLoader = DataLoader(TensorDataset(NXTrain), batch_size=BATCH_STAGE_A, shuffle=True, drop_last=True)
    NXFullNp = MakeSeriesWindows(NTrainSeries, WARM + UNROLL, stride=SEQ_STRIDE)
    NXFull = torch.tensor(NXFullNp, dtype=torch.float32, device=DEVICE)
    NXValFullNp = MakeSeriesWindows(NValSeries, WARM + UNROLL, stride=64, max_per_series=8)
    NXValFull = torch.tensor(NXValFullNp, dtype=torch.float32, device=DEVICE)
    NXValWarm, NXValFut = NXValFull[:, :WARM], NXValFull[:, WARM:]

    NScale = float(NHoldoutObs.std())
    NBounds = (float(NTrainFlat.min()), float(NTrainFlat.max()))
    NWarmNp, NFutNp = EvalWindowsFromTrajectories(NHoldoutObs, WARM, HORIZON, per_traj=2, seed=7)
    NWarm = torch.tensor(NWarmNp, dtype=torch.float32, device=DEVICE)
    NFut = torch.tensor(NFutNp, dtype=torch.float32, device=DEVICE)
    NEvalKw = dict(dt=DT, threshold=THRESHOLD, scale=NScale, noise_floor=NNoiseFloor, bounds=NBounds)

    NCkptDir = CHECKPOINT_DIR / "noise_sweep" / f"noise{Noise:g}"
    NCkptDir.mkdir(parents=True, exist_ok=True)

    Saved = (XTrain, Loader, TrainSeries, XFull, XValFull, XValWarm, XValFut, NOISE_FLOOR, CHECKPOINT_DIR)
    try:
        XTrain, Loader, TrainSeries = NXTrain, NLoader, NTrainSeries
        XFull, XValFull, XValWarm, XValFut = NXFull, NXValFull, NXValWarm, NXValFut
        NOISE_FLOOR, CHECKPOINT_DIR = NNoiseFloor, NCkptDir

        NOursModel, _ = TrainOurs(Seed, Verbose=False)
        NJoint01, _ = TrainJoint(0.1, Seed=Seed)
        NJoint05, _ = TrainJoint(0.5, Seed=Seed)
        NSigmoid, _ = TrainSigmoid(Phi=0.5, Seed=Seed)
    finally:
        (XTrain, Loader, TrainSeries, XFull, XValFull, XValWarm, XValFut,
         NOISE_FLOOR, CHECKPOINT_DIR) = Saved

    def NAgg(Model, Label):
        # Agg() itself reads the GLOBAL Warm/Fut/EvalKw, which is exactly what
        # must NOT happen here -- this noise level has its own. Single seed,
        # so no averaging is needed, just EvaluateModel wrapped in the same
        # dict shape Agg produces (n_seeds, *_std) so downstream code that
        # reads either kind of result works unchanged.
        R = EvaluateModel(Model, NWarm, NFut, **NEvalKw)
        Out = {"model": Label, "n_seeds": 1}
        for k, v in R.items():
            if k == "curve":
                Out["curve"] = v
            elif isinstance(v, (int, float, bool)):
                Out[k] = float(v)
                Out[k + "_std"] = 0.0
        return Out

    NResults = {}
    NResults["Ours (two-stage)"] = NAgg(NOursModel, "Ours (two-stage)")
    NResults["phi=0.1"] = NAgg(NJoint01, "phi=0.1")
    NResults["phi=0.5"] = NAgg(NJoint05, "phi=0.5")
    NResults["AEGRU+sigmoid (D)"] = NAgg(NSigmoid, "AEGRU+sigmoid (D)")
    for Name, Pred in [("persistence", PersistenceRollout(NWarmNp, HORIZON)),
                       ("climatology", MeanRollout(NWarmNp, HORIZON, NTrainFlat.mean(0)))]:
        Fm = ForecastMetrics(NFutNp, Pred, DT, THRESHOLD, NScale, NBounds)
        NResults[Name] = {"model": Name, "recon_nrmse": np.nan, "recon_mse": np.nan,
                          "recon_r2": np.nan, **Fm}
    return NResults, NNoiseFloor


NoiseSweepResults = {0.05: Results}   # already-seeded headline results, reused not re-run
NoiseSweepFloors = {0.05: NOISE_FLOOR}
for _Noise in tqdm(NOISE_SWEEP_LEVELS, desc="noise sweep"):
    NoiseSweepResults[_Noise], NoiseSweepFloors[_Noise] = RunNoiseLevel(_Noise)
    print(f"noise={_Noise:g} done")


## Noise sweep summary

In [ ]:
_SweepModels = ["Ours (two-stage)", "phi=0.1", "phi=0.5", "AEGRU+sigmoid (D)",
               "persistence", "climatology"]

_MissingRow = {"vpt": np.nan, "recon_mse": np.nan, "fcst_nrmse_full": np.nan, "divergence": np.nan}

SweepTable = pd.DataFrame([
    {"noise": Noise, "model": Name,
     "recon x floor": (r.get("recon_mse") / NoiseSweepFloors[Noise]
                       if r.get("recon_mse") is not None and not np.isnan(r.get("recon_mse", np.nan))
                       else np.nan),
     "VPT (Lyap)": r.get("vpt"), "NRMSE @ 5LT": r.get("fcst_nrmse_full"),
     "divergence": r.get("divergence")}
    for Noise in [0.05] + NOISE_SWEEP_LEVELS
    for Name in _SweepModels
    for r in [NoiseSweepResults[Noise].get(Name, {"model": Name, **_MissingRow})]
]).round(4)
SweepTable



### Does the gap between Ours and the best phi widen, narrow, or hold?

In [ ]:
print("VPT gap: Ours minus the better of phi=0.1/phi=0.5, at each noise level")
_Gaps = []
for Noise in [0.05] + NOISE_SWEEP_LEVELS:
    R = NoiseSweepResults[Noise]
    OursVpt = R["Ours (two-stage)"]["vpt"]
    PhiVpts = [R[k]["vpt"] for k in ("phi=0.1", "phi=0.5") if k in R]
    BestPhiVpt = max(PhiVpts) if PhiVpts else np.nan
    Gap = OursVpt - BestPhiVpt
    _Gaps.append(Gap)
    print(f"  noise={Noise:g}: Ours {OursVpt:.4f}  best phi {BestPhiVpt:.4f}  gap {Gap:+.4f}")

if _Gaps[-1] > _Gaps[0] and all(b >= a - 1e-9 for a, b in zip(_Gaps, _Gaps[1:])):
    print("\nDIRECTION: the gap WIDENS as noise increases (monotonically, 0.05 -> 0.15 -> 0.30).")
elif _Gaps[-1] < _Gaps[0] and all(b <= a + 1e-9 for a, b in zip(_Gaps, _Gaps[1:])):
    print("\nDIRECTION: the gap NARROWS as noise increases (monotonically, 0.05 -> 0.15 -> 0.30).")
else:
    print(f"\nDIRECTION: not monotonic -- gap sequence is {[round(g, 4) for g in _Gaps]} "
         f"across noise {[0.05] + NOISE_SWEEP_LEVELS}. Reporting the sequence rather than a single word.")


# Paper figures

Regenerates the figures used in the write-up into `Paper/figs/`, from the same
`Results` dict every table above is built from --- so the paper cannot drift
from the notebook. Titles are omitted deliberately: in the paper the LaTeX
caption carries the title, and a baked-in matplotlib title would duplicate it
and waste vertical space.


In [ ]:
import matplotlib.pyplot as _plt

FIGDIR = ROOT / "Paper" / "figs"
FIGDIR.mkdir(parents=True, exist_ok=True)


# Export at the size the paper actually displays them (~2.6in wide in a
# two-panel row), not larger: a 5in figure scaled down to 2.5in in LaTeX also
# scales its 7pt tick labels down to ~3pt, which is unreadable in print.
def _Save(Fig, Name):
    # constrained layout does not reserve room for a legend anchored outside the
    # axes, so it collides with the x-label at these sizes. Switch the engine off
    # and let bbox_inches="tight" size the canvas around everything instead.
    Fig.set_layout_engine("none")
    Fig.subplots_adjust(left=0.20, right=0.97, top=0.96, bottom=0.20)
    Path = FIGDIR / Name
    Fig.savefig(Path, dpi=300, bbox_inches="tight")
    _plt.close(Fig)
    print(f"  {Name}")


# -- Figure 1: the frontier plot, no title, legend below to keep it narrow.
# Shorter axis labels than the notebook default: at paper column width the
# long ones overrun the tight bounding box and get clipped.
Fig, Ax = PlotParetoFront(Frontier, Points, NoiseFloor=FloorNrmse,
                          Width=2.9, Height=2.3, Title=None,
                          XLabel="Reconstruction NRMSE (lower)",
                          YLabel="VPT, Lyapunov times (higher)")
Ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=5.5,
          frameon=True)
_Save(Fig, "paper_pareto.png")

# -- Figure 2: horizon curves, restricted to the models actually discussed in
# the text. Plotting all 14 makes a legend wider than the axes and buries the
# one thing this panel is for: which curves explode and which saturate.
PaperCurves = {k: Results[k]["curve"] for k in
               ["Ours (two-stage)", "AEGRU+sigmoid (D)", "PINN (true physics)",
                "phi=0.5", "climatology"] if k in Results}
Fig, Ax = PlotHorizonCurves(PaperCurves, Dt=DT, Lam=LAMBDA_MAX,
                            Threshold=THRESHOLD, Width=2.9, Height=2.3, Title=None)
Ax.set_yscale("log")
Ax.set_ylabel("NRMSE (log scale)")
Ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=5.5)
_Save(Fig, "paper_horizon.png")

print("paper figures written to", FIGDIR)
